### Unity Catalog

In [0]:
%sql
drop catalog if exists catalog_nbfc cascade;

In [0]:

%sql

-- 1. Create a catalog to organize NBFC data
CREATE CATALOG IF NOT EXISTS catalog_nbfc;

-- 2. Create a schema to organize loan-related tables
CREATE SCHEMA IF NOT EXISTS catalog_nbfc.schema_loan;

-- 3. Create the loan details table
CREATE TABLE IF NOT EXISTS catalog_nbfc.schema_loan.loan_detail (
    loan_id STRING,      
    loan_amount INT,      
    adhaar_card STRING,   
    regions STRING,       
    salary INT            
);
INSERT INTO catalog_nbfc.schema_loan.loan_detail
VALUES
    ('L001', 500000, '123456789012', 'North', 45000),
    ('L002', 750000, '234567890123', 'South', 60000),
    ('L003', 300000, '345678901234', 'East', 35000),
    ('L004', 900000, '456789012345', 'West', 80000),
    ('L005', 450000, '567890123456', 'North', 50000);


-- 5. Create the employee table
CREATE TABLE IF NOT EXISTS catalog_nbfc.schema_loan.employee (
    employee_id INT,       
    employee_name STRING,  
    department STRING,     
    loan_id STRING,        
    salary INT             
);
INSERT INTO catalog_nbfc.schema_loan.employee
VALUES
    (101, 'Amit', 'Finance', 'L001', 45000),
    (102, 'Priya', 'IT', 'L002', 60000),
    (103, 'Rahul', 'HR', 'L003', 35000),
    (104, 'Neha', 'Finance', 'L006', 50000);

-- 7. Create a user roles table for role-based application logic
CREATE TABLE IF NOT EXISTS catalog_nbfc.schema_loan.user_roles (
    username STRING,  
    role STRING  ,
    regions STRING     
);

-- 8. Insert sample users and their roles
INSERT INTO catalog_nbfc.schema_loan.user_roles
VALUES
    ('admin@gmail.com', 'admin','North'),
    ('sonaliagrahari08@gmail.com', 'engineer','South'),
    ('manager@example.com', 'manager','East');


num_affected_rows,num_inserted_rows
3,3


### 1. Create a masking function on Columns

In [0]:
%sql

-- 1. Mask Aadhaar for non-admin users
CREATE OR REPLACE FUNCTION catalog_nbfc.schema_loan.mask_pii(value STRING)
RETURNS STRING
RETURN CASE
    WHEN EXISTS (
        SELECT 1
        FROM catalog_nbfc.schema_loan.user_roles r
        WHERE LOWER(r.username) = LOWER(session_user())
          AND LOWER(r.role) = 'admin'
    )
    THEN value
    ELSE 'XXXXXXXXXXXX'
END;

-- 2. Mask salary for non-admin users
CREATE OR REPLACE FUNCTION catalog_nbfc.schema_loan.mask_salary(value INT)
RETURNS INT
RETURN CASE
    WHEN EXISTS (
        SELECT 1
        FROM catalog_nbfc.schema_loan.user_roles r
        WHERE LOWER(r.username) = LOWER(session_user())
          AND LOWER(r.role) = 'admin'
    )
    THEN value
    ELSE NULL
END;

-- 3. Apply Aadhaar masking
ALTER TABLE catalog_nbfc.schema_loan.loan_detail
ALTER COLUMN adhaar_card
SET MASK catalog_nbfc.schema_loan.mask_pii;

-- 4. Apply salary masking
ALTER TABLE catalog_nbfc.schema_loan.loan_detail
ALTER COLUMN salary
SET MASK catalog_nbfc.schema_loan.mask_salary;

In [0]:
%sql
select * from catalog_nbfc.schema_loan.loan_detail

loan_id,loan_amount,adhaar_card,regions,salary
L001,500000,XXXXXXXXXXXX,North,null
L002,750000,XXXXXXXXXXXX,South,null
L003,300000,XXXXXXXXXXXX,East,null
L004,900000,XXXXXXXXXXXX,West,null
L005,450000,XXXXXXXXXXXX,North,null


In [0]:
%sql
UPDATE catalog_nbfc.schema_loan.user_roles
SET regions = 'South'
WHERE LOWER(username) = LOWER(session_user())
  AND LOWER(role) = 'engineer';


CREATE OR REPLACE FUNCTION
catalog_nbfc.schema_loan.filter_loan_region(row_region STRING)
RETURNS BOOLEAN
RETURN EXISTS (
    SELECT 1
    FROM catalog_nbfc.schema_loan.user_roles u
    WHERE LOWER(u.username) = LOWER(session_user())
      AND (
          LOWER(u.role) = 'engineer'
          OR UPPER(u.regions) = UPPER(row_region)
      )
);

ALTER TABLE catalog_nbfc.schema_loan.loan_detail
SET ROW FILTER catalog_nbfc.schema_loan.filter_loan_region
ON (regions);

In [0]:
%sql
select * from catalog_nbfc.schema_loan.loan_detail

loan_id,loan_amount,adhaar_card,regions,salary
L001,500000,XXXXXXXXXXXX,North,null
L002,750000,XXXXXXXXXXXX,South,null
L003,300000,XXXXXXXXXXXX,East,null
L004,900000,XXXXXXXXXXXX,West,null
L005,450000,XXXXXXXXXXXX,North,null
